[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-05-detection-segmentation-gradcam-fgsm.ipynb)

# Advanced Discussion 5 — boxes, masks, heat-maps and adversarial inputs

Reproduces the chapter 5 advanced discussion: parameter counting for 1×1 bottlenecks, IoU and non-maximum suppression, detection and segmentation metrics, Grad-CAM, and the FGSM attack.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: the detector is created without pretrained weights, so nothing is downloaded. Use pretrained weights in Lab 5 for real results.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, torchvision
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.manual_seed(0); np.random.seed(0)

## 1. parameter counting

In [ ]:
def n_params(m): return sum(p.numel() for p in m.parameters())
plain = nn.Conv2d(256, 256, 3, padding=1, bias=False)
bott = nn.Sequential(nn.Conv2d(256, 64, 1, bias=False), nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.Conv2d(64, 256, 1, bias=False))
print("plain 3x3 256->256:", n_params(plain), " bottleneck:", n_params(bott), " ratio %.1fx" % (n_params(plain) / n_params(bott)))
print("MLP first layer 224x224x3 -> 512:", 224 * 224 * 3 * 512, " conv 3x3x3->64:", 3 * 3 * 3 * 64 + 64)

## 2. IoU + NMS

In [ ]:
def iou(a, b):
    """a, b: (x1, y1, x2, y2)"""
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def nms(boxes, scores, thr=0.5):
    order = sorted(range(len(boxes)), key=lambda i: -scores[i]); keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < thr]
    return keep

boxes = [(10, 10, 60, 60), (12, 12, 62, 64), (100, 100, 150, 150), (14, 8, 58, 58)]
scores = [0.90, 0.80, 0.75, 0.60]
print("iou(box0, box1) = %.3f" % iou(boxes[0], boxes[1]), " nms keep:", nms(boxes, scores))
tb = torchvision.ops.nms(torch.tensor(boxes, dtype=torch.float32), torch.tensor(scores), 0.5)
print("torchvision nms:", tb.tolist())

## 3. torchvision detection model API (no weights, so no download)

In [ ]:
det = torchvision.models.detection.fasterrcnn_mobilenet_v3_large_320_fpn(weights=None, weights_backbone=None, num_classes=3)
det.eval()
with torch.no_grad(): out = det([torch.rand(3, 240, 320)])
print("detector output keys:", sorted(out[0].keys()))

## 4. dice / iou for masks

In [ ]:
def dice(pred, gt, eps=1e-7):
    inter = (pred & gt).sum().item(); return (2 * inter + eps) / (pred.sum().item() + gt.sum().item() + eps)
def miou(pred, gt, eps=1e-7):
    inter = (pred & gt).sum().item(); union = (pred | gt).sum().item(); return (inter + eps) / (union + eps)
gt = torch.zeros(10, 10, dtype=torch.bool); gt[2:6, 2:6] = True
pr = torch.zeros(10, 10, dtype=torch.bool); pr[3:7, 3:7] = True
print("dice %.3f iou %.3f pixel-acc %.2f" % (dice(pr, gt), miou(pr, gt), (pr == gt).float().mean().item()))
gt2 = torch.zeros(100, 100, dtype=torch.bool); gt2[40:44, 40:44] = True
allbg = torch.zeros(100, 100, dtype=torch.bool)
print("all-background prediction: pixel-acc %.4f, dice %.4f" % ((allbg == gt2).float().mean().item(), dice(allbg, gt2)))

## 5. small CNN on digits + Grad-CAM + FGSM

In [ ]:
X, y = load_digits(return_X_y=True)
X = torch.tensor(X / 16.0, dtype=torch.float32).view(-1, 1, 8, 8); y = torch.tensor(y)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y.numpy())
class Net(nn.Module):
    def __init__(s):
        super().__init__()
        s.c1 = nn.Conv2d(1, 16, 3, padding=1); s.c2 = nn.Conv2d(16, 32, 3, padding=1); s.fc = nn.Linear(32, 10)
    def forward(s, x):
        x = F.relu(s.c1(x)); x = F.relu(s.c2(x)); s.fmap = x
        return s.fc(x.mean((2, 3)))
net = Net(); opt = torch.optim.Adam(net.parameters(), lr=3e-3)
for ep in range(60):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        idx = perm[i:i + 64]; opt.zero_grad(); F.cross_entropy(net(Xtr[idx]), ytr[idx]).backward(); opt.step()
net.eval()
acc = (net(Xte).argmax(1) == yte).float().mean().item(); print("cnn test acc %.3f" % acc)

def grad_cam(model, x, cls=None):
    model.eval(); logits = model(x); fmap = model.fmap; fmap.retain_grad()
    cls = logits.argmax(1) if cls is None else cls
    logits[0, cls].backward()
    w = fmap.grad.mean((2, 3), keepdim=True)
    cam = F.relu((w * fmap).sum(1)).detach()[0]
    return cam / (cam.max() + 1e-8)
cam = grad_cam(net, Xte[:1]); print("grad-cam shape", tuple(cam.shape), "max %.2f" % cam.max().item())

def fgsm(model, x, y, eps):
    x = x.clone().requires_grad_(True); F.cross_entropy(model(x), y).backward()
    return (x + eps * x.grad.sign()).clamp(0, 1).detach()
for eps in (0.0, 0.05, 0.1, 0.2, 0.3):
    xa = fgsm(net, Xte, yte, eps); a = (net(xa).argmax(1) == yte).float().mean().item()
    print("eps %.2f -> adversarial accuracy %.3f" % (eps, a))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.